# EmbeddingGemma 2 ile benzer mekanlar

Takım 5 · Alanya turizm rehberi · Adım 12.3

Bu not defteri `veri.json` içindeki mekanlar için en benzer 3 mekanı bulur ve `benzerler.json` dosyasına yazar.

**Uyarı:** float16 kullanmayın; hata vermeden bozuk sonuç üretebilir. Birkaç yüz kayıt için Colab'ın CPU'su yeter.

In [ ]:
!pip install -U sentence-transformers

## 1. veri.json dosyasını yükleyin

Dosyada her kaydın `id` ve `metin` alanı olmalı. Tüm `veri.json` yüklenirse yalnızca `mekanlar` listesi kullanılır.

In [ ]:
from google.colab import files
uploaded = files.upload()  # veri.json dosyasını seçin

## 2. Modeli yükleyin ve benzerleri hesaplayın

In [ ]:
import json, torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-2",
                            model_kwargs={"torch_dtype": torch.float32})

veri = json.load(open("veri.json", encoding="utf-8"))
kayitlar = veri["mekanlar"] if isinstance(veri, dict) else veri
print(len(kayitlar), "kayıt okundu")

emb = model.encode([k["metin"] for k in kayitlar], prompt_name="Document")
benzerlik = model.similarity(emb, emb)

sonuc = {}
for i, k in enumerate(kayitlar):
    skor = benzerlik[i].clone()
    skor[i] = -1  # kaydın kendisini atla
    en_iyi = torch.topk(skor, k=3).indices.tolist()
    sonuc[k["id"]] = [kayitlar[j]["id"] for j in en_iyi]

json.dump(sonuc, open("benzerler.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("benzerler.json yazıldı")

## 3. Sonuçları gözle kontrol edin

Benzer çıkan mekanlar mantıklı mı? Mantıksız çiftleri takıma bildirin; gerekirse mekan metnini düzeltip bu not defterini yeniden çalıştırın.

In [ ]:
ad = {k["id"]: k.get("ad", k["id"]) for k in kayitlar}
for kimlik, liste in list(sonuc.items())[:10]:
    print(ad[kimlik], "->", ", ".join(ad[x] for x in liste))

## 4. Dosyayı indirin

İndirilen `benzerler.json` dosyasını repoda `src/data/` altına koyun. Bu not defterini Dosya > İndir > .ipynb ile indirip `notebooks/embeddinggemma.ipynb` olarak ekleyin.

In [ ]:
files.download("benzerler.json")